<h1 align="center">📐 First-Order Logic: Formalization, Unification & Resolution</h1>

<p align="center"><i>Talk about objects and relations with quantifiers, make terms match with unification, and prove theorems by refutation, all in plain Python.</i></p>

<p align="center">
  <img src="https://img.shields.io/badge/Course-Knowledge%20Representation%20%26%20Reasoning-6f42c1" />
  <img src="https://img.shields.io/badge/Python-3.x-blue" />
  <img src="https://img.shields.io/badge/Level-Intermediate-yellow" />
</p>

<p align="center">
  <a href="https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/1-first-order-logic/first_order_logic.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</p>

---

## 📌 At a Glance

| | |
|---|---|
| **Topics** | Predicates · Quantifiers · English→FOL · Finite models · Substitution · MGU · Occurs-check · Resolution refutation |
| **Libraries** | none for the logic itself, `sympy` for one cross-check |
| **Run time** | Under 1 minute |

## 🗂️ Contents
1. [Formalizing a Domain](#part-1)
2. [Unification & Substitution](#part-2)
3. [Resolution Refutation](#part-3)
4. [Summary](#summary)

---

**Setup** (only needed once, e.g. on a fresh Colab runtime):

In [1]:
!pip install -q sympy

<a id="part-1"></a>

## 📚 Part 1 · Formalizing a Domain

> 🎯 **Goal:** Translate English into FOL, then *check* the formulas against a small model.

FOL extends propositional logic with **predicates**, **functions** and the quantifiers $\forall$ (for all) and $\exists$ (there exists).

### 1.1 English → FOL

| English | First-Order Logic | Predicates / quantifiers |
|---|---|---|
| All humans are mortal | $\forall x\,(Human(x) \rightarrow Mortal(x))$ | `Human/1`, `Mortal/1`, $\forall$ |
| Socrates is a human | $Human(socrates)$ | constant `socrates` |
| Some student loves AI | $\exists x\,(Student(x) \land Loves(x, ai))$ | `Student/1`, `Loves/2`, $\exists$ |
| Every dog has an owner | $\forall x\,(Dog(x) \rightarrow \exists y\,Owns(y, x))$ | nested quantifiers |
| No fish is a mammal | $\forall x\,(Fish(x) \rightarrow \neg Mammal(x))$ | $\forall$ with negation |

Rule of thumb: $\forall$ pairs with $\rightarrow$, and $\exists$ pairs with $\land$.

### 1.2 Evaluate the formulas in a finite model
A **model** gives a domain of objects and says which predicates hold for which objects. In Python, $\forall$ is `all(...)`, $\exists$ is `any(...)`, and $A \rightarrow B$ is `(not A) or B`.

In [2]:
domain  = ['socrates', 'plato', 'ali', 'rex', 'salmon', 'ai']

Human   = {'socrates', 'plato', 'ali'}
Student = {'ali'}
Mortal  = {'socrates', 'plato', 'ali', 'rex', 'salmon'}
Dog     = {'rex'}
Fish    = {'salmon'}
Mammal  = {'rex'}
Loves   = {('ali', 'ai')}
Owns    = {('ali', 'rex')}

def implies(a, b): return (not a) or b
def forall(pred):  return all(pred(x) for x in domain)
def exists(pred):  return any(pred(x) for x in domain)

print('forall x (Human(x) -> Mortal(x))          :',
      forall(lambda x: implies(x in Human, x in Mortal)))
print('exists x (Student(x) & Loves(x, ai))      :',
      exists(lambda x: x in Student and (x, 'ai') in Loves))
print('forall x (Dog(x) -> exists y Owns(y, x)) :',
      forall(lambda x: implies(x in Dog, exists(lambda y: (y, x) in Owns))))
print('forall x (Fish(x) -> not Mammal(x))      :',
      forall(lambda x: implies(x in Fish, x not in Mammal)))

forall x (Human(x) -> Mortal(x))          : True
exists x (Student(x) & Loves(x, ai))      : True
forall x (Dog(x) -> exists y Owns(y, x)) : True
forall x (Fish(x) -> not Mammal(x))      : True

### 1.3 Find a counterexample
A universal statement is false as soon as **one** object breaks it. Let us make the salmon a mammal.

In [3]:
Mammal_bad = Mammal | {'salmon'}

violators = [x for x in domain if not implies(x in Fish, x not in Mammal_bad)]
print('forall x (Fish(x) -> not Mammal(x)) holds?', not violators)
print('Counterexample(s):', violators)

forall x (Fish(x) -> not Mammal(x)) holds? False
Counterexample(s): ['salmon']

### 1.4 Practice: formalize and check
Add a university domain to the model and test: *"Every student is enrolled in some course"*, $\forall x\,(Student(x) \rightarrow \exists c\,Enrolled(x, c))$.

In [4]:
Enrolled = {('ali', 'krr'), ('sara', 'krr')}
Student2 = {'ali', 'sara', 'bilal'}
courses  = ['krr', 'ai']

holds = all(any((s, c) in Enrolled for c in courses) for s in Student2)
print('Every student is enrolled in some course:', holds)
print('Students with no enrolment:', [s for s in sorted(Student2)
                                      if not any((s, c) in Enrolled for c in courses)])

Every student is enrolled in some course: False
Students with no enrolment: ['bilal']

> 💡 **Key Takeaway:** $\forall$ means *every* object satisfies the condition (one counterexample refutes it); $\exists$ means *at least one* does (one witness proves it).

---
<a id="part-2"></a>

## 🔗 Part 2 · Unification & Substitution

> 🎯 **Goal:** Find a substitution that makes two terms identical, and compute the **most general unifier (MGU)**.

A **substitution** replaces variables by terms, e.g. $\{X/ali,\ Y/ai\}$. Unification is the engine behind resolution and Prolog.

### 2.1 The algorithm
Terms are tuples (`('loves', 'X', 'ai')`); strings starting with a capital letter are variables.

In [5]:
def is_var(t):
    return isinstance(t, str) and t[0].isupper()

def occurs(var, t, subst):
    """Does var occur inside t? (occurs-check)"""
    if t == var:
        return True
    if is_var(t) and t in subst:
        return occurs(var, subst[t], subst)
    if isinstance(t, tuple):
        return any(occurs(var, a, subst) for a in t)
    return False

def unify(x, y, subst):
    if subst is None:
        return None
    if x == y:
        return subst
    if is_var(x):
        return unify_var(x, y, subst)
    if is_var(y):
        return unify_var(y, x, subst)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            subst = unify(a, b, subst)
        return subst
    return None

def unify_var(var, x, subst):
    if var in subst:
        return unify(subst[var], x, subst)
    if occurs(var, x, subst):
        return None
    subst = dict(subst)
    subst[var] = x
    return subst

def apply(subst, t):
    """Apply a substitution to a term."""
    if is_var(t):
        return apply(subst, subst[t]) if t in subst else t
    if isinstance(t, tuple):
        return tuple(apply(subst, a) for a in t)
    return t

# loves(X, ai)  vs  loves(ali, Y)
t1 = ('loves', 'X', 'ai')
t2 = ('loves', 'ali', 'Y')
mgu = unify(t1, t2, {})
print('MGU        :', mgu)
print('t1 after   :', apply(mgu, t1))
print('t2 after   :', apply(mgu, t2))

MGU        : {'X': 'ali', 'Y': 'ai'}
t1 after   : ('loves', 'ali', 'ai')
t2 after   : ('loves', 'ali', 'ai')

### 2.2 Practice tasks

In [6]:
def show(t1, t2):
    s = unify(t1, t2, {})
    print(f'{t1}  ~  {t2}')
    if s is None:
        print('   FAIL (no unifier)')
    else:
        print('   MGU   :', s)
        print('   result:', apply(s, t1))

# 1. knows(X, X) vs knows(ali, Y): X=ali, then the repeated X forces Y=ali
show(('knows', 'X', 'X'), ('knows', 'ali', 'Y'))

# 2. a failing case: two different constants never unify
show(('loves', 'ali', 'ai'), ('loves', 'bob', 'ai'))

# 3. nested terms: f(X, g(Y)) vs f(a, g(b))
show(('f', 'X', ('g', 'Y')), ('f', 'a', ('g', 'b')))

# 4. occurs-check: X vs f(X) would need an infinite term
show('X', ('f', 'X'))

('knows', 'X', 'X')  ~  ('knows', 'ali', 'Y')
   MGU   : {'X': 'ali', 'Y': 'ali'}
   result: ('knows', 'ali', 'ali')
('loves', 'ali', 'ai')  ~  ('loves', 'bob', 'ai')
   FAIL (no unifier)
('f', 'X', ('g', 'Y'))  ~  ('f', 'a', ('g', 'b'))
   MGU   : {'X': 'a', 'Y': 'b'}
   result: ('f', 'a', ('g', 'b'))
X  ~  ('f', 'X')
   FAIL (no unifier)

> 💡 **Key Takeaway:** The **MGU** is the least-committal substitution that makes two terms equal. Constants clash, variables bind, and the occurs-check stops infinite terms.

---
<a id="part-3"></a>

## 🔬 Part 3 · Resolution Refutation

> 🎯 **Goal:** Prove $KB \models \alpha$ by adding $\neg\alpha$ and deriving the **empty clause** $\square$.

Clauses are Python sets of literals; negation uses a `~` prefix. Each resolution step combines two clauses containing a literal and its negation, cancelling them.

In [7]:
def negate(lit):
    return lit[1:] if lit.startswith('~') else '~' + lit

def resolve(ci, cj):
    resolvents = []
    for lit in ci:
        if negate(lit) in cj:
            new = (ci - {lit}) | (cj - {negate(lit)})
            resolvents.append(frozenset(new))
    return resolvents

def resolution(kb, query):
    clauses = set(kb) | {frozenset({negate(query)})}
    while True:
        new = set()
        pairs = [(a, b) for a in clauses for b in clauses if a != b]
        for a, b in pairs:
            for r in resolve(a, b):
                if len(r) == 0:
                    return True          # empty clause -> proved
                new.add(r)
        if new.issubset(clauses):
            return False                 # nothing new -> no proof
        clauses |= new

# KB: (P -> Q) and P, as clauses. Prove Q.
kb = [frozenset({'~P', 'Q'}), frozenset({'P'})]
print('KB entails Q:', resolution(kb, 'Q'))

KB entails Q: True

### 3.1 Practice tasks

In [8]:
C = lambda *lits: frozenset(lits)

print('1. {P v Q, ~P} entails Q            :', resolution([C('P', 'Q'), C('~P')], 'Q'))
print('2. {P, P->Q, Q->R} entails R        :', resolution([C('P'), C('~P', 'Q'), C('~Q', 'R')], 'R'))
print('3. {P v Q} entails P  (no proof)    :', resolution([C('P', 'Q')], 'P'))
print('4. {P->Q, ~Q} entails ~P (Tollens)  :', resolution([C('~P', 'Q'), C('~Q')], '~P'))

1. {P v Q, ~P} entails Q            : True
2. {P, P->Q, Q->R} entails R        : True
3. {P v Q} entails P  (no proof)    : False
4. {P->Q, ~Q} entails ~P (Tollens)  : True

### 3.2 Cross-check with SymPy

In [9]:
from sympy import symbols
from sympy.logic.inference import entails

P, Q, R = symbols('P Q R')
print('{P | Q, ~P}     |= Q  :', entails(Q, [P | Q, ~P]))
print('{P, P>>Q, Q>>R} |= R  :', entails(R, [P, P >> Q, Q >> R]))
print('{P | Q}         |= P  :', entails(P, [P | Q]))
print('{P>>Q, ~Q}      |= ~P :', entails(~P, [P >> Q, ~Q]))

{P | Q, ~P}     |= Q  : True
{P, P>>Q, Q>>R} |= R  : True
{P | Q}         |= P  : False
{P>>Q, ~Q}      |= ~P : True

Both methods agree. Task 3 returns **False**: $P = false,\ Q = true$ satisfies the KB but falsifies the query, so no refutation exists.

> 💡 **Key Takeaway:** **Refutation** turns "is it entailed?" into "can we derive a contradiction?". Reaching $\square$ proves entailment; saturating without $\square$ proves non-entailment.

---
<a id="summary"></a>
## ✅ Summary

| Part | Topic | What you learned |
|---|---|---|
| 1 | Formalizing a domain | Predicates, quantifiers, English→FOL, checking formulas in a model, counterexamples |
| 2 | Unification | Substitution, MGU, occurs-check |
| 3 | Resolution | Clause form, refutation, empty clause, entailment / non-entailment |

<p align="center"><i>⬅️ Back to Knowledge Bases & Propositional Logic · Next: Prolog ➡️</i></p>